# PJM Load Forecast v2 — Forecast Evolution by Rank

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add backend to path so we can import the utils
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))

from src.utils.azure_postgresql import pull_from_db

INFO:root:CONFIG_DIR: c:\Users\AidanKeaveny\Documents\github\helioscta-pjm-da\backend\src


In [2]:
# Pull forecast data for current date
sql_path = Path.cwd() / "pjm_load_forecast_v2.sql"
query = sql_path.read_text()

df = pull_from_db(query=query)
print(f"{len(df):,} rows")
print(f"Forecast date: {df['forecast_date'].unique()}")
print(f"Regions: {sorted(df['region'].unique())}")
print(f"Ranks: {sorted(df['forecast_rank'].unique())}")
df.head()

14,592 rows
Forecast date: [datetime.date(2026, 3, 4)]
Regions: ['MIDATL', 'RTO', 'SOUTH', 'WEST']
Ranks: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int

,datetime,forecast_rank,forecast_execution_datetime,forecast_execution_date,forecast_date,hour_ending,region,forecast_load_mw
0,2026-03-04 00:00:00,1,2026-03-04 10:47:17,2026-03-04,2026-03-04,1,MIDATL,29224.0
1,2026-03-04 01:00:00,1,2026-03-04 10:47:17,2026-03-04,2026-03-04,2,MIDATL,28370.0
2,2026-03-04 02:00:00,1,2026-03-04 10:47:17,2026-03-04,2026-03-04,3,MIDATL,27817.0
3,2026-03-04 03:00:00,1,2026-03-04 10:47:17,2026-03-04,2026-03-04,4,MIDATL,27689.0
4,2026-03-04 04:00:00,1,2026-03-04 10:47:17,2026-03-04,2026-03-04,5,MIDATL,28248.0


## Forecast Evolution — All Regions
Each line is a forecast vintage (rank). Rank 1 = most recent, higher ranks = older vintages.

In [3]:
regions = sorted(df["region"].unique())

fig = make_subplots(
    rows=len(regions), cols=1,
    shared_xaxes=True,
    subplot_titles=regions,
    vertical_spacing=0.06,
)

for i, region in enumerate(regions, start=1):
    df_region = df[df["region"] == region]
    for rank in sorted(df_region["forecast_rank"].unique()):
        df_rank = df_region[df_region["forecast_rank"] == rank]
        exec_date = df_rank["forecast_execution_date"].iloc[0]
        label = f"Rank {rank} ({exec_date})"
        fig.add_trace(
            go.Scatter(
                x=df_rank["hour_ending"],
                y=df_rank["forecast_load_mw"],
                name=label,
                legendgroup=str(rank),
                showlegend=(i == 1),
                opacity=1.0 if rank == 1 else max(0.3, 1.0 - rank * 0.1),
                line=dict(width=3 if rank == 1 else 1.5),
            ),
            row=i, col=1,
        )
    fig.update_yaxes(title_text="MW", row=i, col=1)

fig.update_xaxes(title_text="Hour Ending", row=len(regions), col=1)
fig.update_layout(
    title=f"Load Forecast Evolution — {df['forecast_date'].iloc[0]}",
    template="plotly_dark",
    height=350 * len(regions),
)
fig.show()

## Per-Region Detail

In [4]:
for region in regions:
    df_region = df[df["region"] == region].copy()
    df_region["rank_label"] = "Rank " + df_region["forecast_rank"].astype(str)
    fig = px.line(
        df_region,
        x="hour_ending",
        y="forecast_load_mw",
        color="rank_label",
        hover_data=["forecast_execution_datetime"],
        title=f"Forecast Evolution — {region} ({df_region['forecast_date'].iloc[0]})",
        labels={"forecast_load_mw": "Forecast Load (MW)", "hour_ending": "Hour Ending", "rank_label": "Vintage"},
    )
    fig.update_layout(template="plotly_dark", height=450)
    fig.show()

## Vintage-to-Vintage Delta (MW change from previous rank)

In [5]:
# Compute MW change between consecutive ranks (newer - older)
df_sorted = df.sort_values(["region", "hour_ending", "forecast_rank"])
df_sorted["prev_load"] = df_sorted.groupby(["region", "hour_ending"])["forecast_load_mw"].shift(1)
df_sorted["delta_mw"] = df_sorted["forecast_load_mw"] - df_sorted["prev_load"]
df_delta = df_sorted.dropna(subset=["delta_mw"])

for region in regions:
    df_region = df_delta[df_delta["region"] == region].copy()
    df_region["rank_label"] = "Rank " + df_region["forecast_rank"].astype(str) + " vs " + (df_region["forecast_rank"] - 1).astype(int).astype(str)
    fig = px.bar(
        df_region,
        x="hour_ending",
        y="delta_mw",
        color="rank_label",
        barmode="group",
        title=f"Vintage Delta — {region} ({df_region['forecast_date'].iloc[0]})",
        labels={"delta_mw": "Delta MW", "hour_ending": "Hour Ending", "rank_label": "Comparison"},
    )
    fig.update_layout(template="plotly_dark", height=400)
    fig.show()

## Summary — Rank 1 vs Oldest Rank (Total Revision)

In [6]:
# Compare newest (rank 1) vs oldest rank to show total revision
max_rank = df["forecast_rank"].max()
df_newest = df[df["forecast_rank"] == 1].set_index(["region", "hour_ending"])["forecast_load_mw"]
df_oldest = df[df["forecast_rank"] == max_rank].set_index(["region", "hour_ending"])["forecast_load_mw"]

df_revision = (df_newest - df_oldest).reset_index()
df_revision.columns = ["region", "hour_ending", "total_revision_mw"]

fig = px.bar(
    df_revision,
    x="hour_ending",
    y="total_revision_mw",
    color="region",
    barmode="group",
    title=f"Total Forecast Revision (Rank 1 vs Rank {max_rank}) — {df['forecast_date'].iloc[0]}",
    labels={"total_revision_mw": "Revision (MW)", "hour_ending": "Hour Ending"},
)
fig.update_layout(template="plotly_dark", height=500)
fig.show()